In [3]:
from collections.abc import Callable
from typing import Any

import numpy as np

from agent_memories.agent.privacy import token_generation as tg


/Users/YasmineFrizlen/Documents/02_Repos/25-26_CE901-SU_CE902-SP_frizlen_yasmine/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "train.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "thedevastator/the-trec-question-classification-dataset-a-longi",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/var/folders/td/c82f_gx522v2zdlnp840x9r00000gn/T/ipykernel_47415/1034878770.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


First 5 records:    label-coarse  label-fine                                               text
0             0           0  How did serfdom develop in and then leave Russ...
1             1           1   What films featured the character Popeye Doyle ?
2             0           0  How can I find a list of celebrities ' real na...
3             1           2  What fowl grabs the spotlight after the Chines...
4             2           3                    What is the full form of .com ?


In [6]:
labels = {0: "Abbreviation", 1: "Description", 2: "Entity", 3: "Person", 4: "Location", 5: "Number"}
df['label-coarse'] = df['label-coarse'].map(labels)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5452 entries, 0 to 5451
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   label-coarse  5452 non-null   str  
 1   label-fine    5452 non-null   int64
 2   text          5452 non-null   str  
dtypes: int64(1), str(2)
memory usage: 444.8 KB


In [8]:
df.head()

,label-coarse,label-fine,text
0,Abbreviation,0,How did serfdom develop in and then leave Russ...
1,Description,1,What films featured the character Popeye Doyle ?
2,Abbreviation,0,How can I find a list of celebrities ' real na...
3,Description,2,What fowl grabs the spotlight after the Chines...
4,Entity,3,What is the full form of .com ?


In [9]:
import importlib

import agent_memories.agent.privacy.privacy_accounting as privacy_accounting

importlib.reload(privacy_accounting)

from agent_memories.agent.privacy.privacy_accounting import (
    PrivacyAccount,
    epsilon_from_rho,
    rho_for,
    solve_r,
)
from agent_memories.agent.privacy.privatisation import (
    sample_private,
    sample_public,
    softmax_l1_distance,
)

In [10]:
def wrap(items: str = " ", *, answer_type: str) -> str:
    return (
        "# [User]\n"
        f"Here are questions with Answer Type: {answer_type}.\n\n"
        "'''\n"
        f"Text: {items}\n"
        "'''\n\n"
        "Please give me another one.\n\n"
        "# [Assistant]\n"
        "'''\n"
        "Question:"
    )

In [ ]:
import html
import torch
from IPython.display import HTML, display

GEMMA_CHUNK_SIZE = 8
MAX_TOTAL_TOKENS = 256
TARGET_EPSILON = 1.0

_PUBLIC_BG = "#c6f6d5"  # green
_PRIVATE_BG = "#fed7d7"  # red


def _laplace(scale: float) -> float:
    return float(np.random.laplace(0.0, scale))


def _stack_logits_microbatched(
    prompt_ids: list[list[int]],
    x_ids: list[int],
    *,
    chunk_size: int,
) -> torch.Tensor:
    rows: list[torch.Tensor] = []
    for start in range(0, len(prompt_ids), chunk_size):
        chunk = prompt_ids[start : start + chunk_size]
        rows.append(tg.get_next_token_logits_from_ids([ids + x_ids for ids in chunk]))
    return torch.cat(rows, dim=0)


def _token_span(tok: int, source: str) -> str:
    bg = _PUBLIC_BG if source == "public" else _PRIVATE_BG
    piece = html.escape(tg.decode([tok])).replace("\n", "<br>")
    return f'<span style="background:{bg}">{piece}</span>'


def show_marked_text(marked_html: str) -> None:
    """Display text with public (green) / private (red) token backgrounds."""
    legend = (
        f'<span style="background:{_PUBLIC_BG}">public</span> &nbsp; '
        f'<span style="background:{_PRIVATE_BG}">private</span>'
    )
    display(HTML(f"<div>{legend}</div><div style='white-space:pre-wrap'>{marked_html}</div>"))


def generate_microbatched(
    texts: list[str],
    *,
    wrap_fn: Callable[..., str],
    s: int,
    c: float,
    tau: float,
    tau_public: float,
    sigma: float,
    theta: float,
    r: int,
    delta: float,
    gemma_chunk_size: int = GEMMA_CHUNK_SIZE,
    max_total_tokens: int = MAX_TOTAL_TOKENS,
    **wrap_kwargs: Any,
) -> tuple[list[str], PrivacyAccount, list[list[str]], list[str]]:
    """Amin Algorithm 1 with chunked Gemma forwards (s can stay at 127 on MPS).

    Emits multiple synthetic examples per batch via the outer ``while t < r``
    loop (Algorithm 1 lines 9--23). Each example starts from an empty suffix.

    Also returns per-example token sources (``"public"`` / ``"private"``)
    and HTML with colored spans for notebook inspection.
    """
    prompts = [wrap_fn(items=text, **wrap_kwargs) for text in texts]
    public_prompt = wrap_fn(**wrap_kwargs)
    prompt_ids = [tg.encode_chat(p) for p in prompts]
    public_ids = tg.encode_chat(public_prompt)

    stop = tg.stop_ids()
    t = 0
    n_public = 0
    theta_hat = theta + _laplace(sigma)
    synthetics: list[str] = []
    all_sources: list[list[str]] = []
    marked_htmls: list[str] = []

    while t < r:
        x_ids: list[int] = []
        sources: list[str] = []
        spans: list[str] = []

        while True:
            if len(x_ids) >= max_total_tokens:
                break

            Z = _stack_logits_microbatched(
                prompt_ids, x_ids, chunk_size=gemma_chunk_size
            )
            z_public = tg.get_next_token_logits_from_ids([public_ids + x_ids])[0]

            d_hat = softmax_l1_distance(Z, z_public, s) + _laplace(2.0 * sigma)
            if d_hat >= theta_hat and t < r:
                tok = sample_private(Z, c, tau, s)
                source = "private"
                t += 1
                theta_hat = theta + _laplace(sigma)
            else:
                tok = sample_public(z_public, tau_public)
                source = "public"
                n_public += 1

            x_ids.append(tok)
            sources.append(source)
            spans.append(_token_span(tok, source))
            if tok in stop:
                break

        synthetics.append(tg.decode(x_ids))
        all_sources.append(sources)
        marked_htmls.append("".join(spans))

    rho = rho_for(r, s, c, tau, sigma)
    account = PrivacyAccount(
        epsilon=epsilon_from_rho(rho, delta),
        delta=delta,
        rho=rho,
        r=r,
        s=s,
        c=c,
        tau=tau,
        sigma=sigma,
        private_tokens_used=t,
        public_tokens_used=n_public,
    )
    return synthetics, account, all_sources, marked_htmls

In [12]:
from sklearn.model_selection import train_test_split

In [ ]:
S = 500

returned_memories = []
train_df, test_df = train_test_split(df, test_size=0.1)
delta = 1.0 / len(train_df)

for category in train_df["label-coarse"].unique():
    print(f"Category: {category}")
    texts = train_df.loc[train_df["label-coarse"] == category, "text"].tolist()
    batches = [texts[i : i + S] for i in range(0, len(texts), S)]
    n_batches = len(batches)
    r = solve_r(
        TARGET_EPSILON, delta, s=S, c=10, tau=1.5, sigma=0.1, r_max=None
    )
    if r == 0:
        print("  budget too tight for category, skipping")
        continue
    for batch_idx, batch in enumerate(batches):
        print(
            f"  batch {batch_idx + 1}/{n_batches} "
            f"(n={len(batch)}, expected_s={S}, r={r})"
        )
        synthetics, account, all_sources, marked_htmls = generate_microbatched(
            batch,
            wrap_fn=wrap,
            answer_type=category,
            s=S,
            c=10,
            tau=1.5,
            theta=0.3,
            sigma=0.1,
            tau_public=1.5,
            r=r,
            delta=delta,
            gemma_chunk_size=GEMMA_CHUNK_SIZE,
        )
        for ex_idx, synthetic in enumerate(synthetics):
            returned_memories.append(
                {
                    "category": category,
                    "batch_idx": batch_idx,
                    "example_idx": ex_idx,
                    "text": synthetic,
                    "token_sources": all_sources[ex_idx],
                    "text_marked_html": marked_htmls[ex_idx],
                    "private_tokens_used": account.private_tokens_used,
                    "n_examples_in_batch": len(synthetics),
                }
            )
            print(
                {
                    "category": category,
                    "batch_idx": batch_idx,
                    "example_idx": ex_idx,
                    "private_tokens_used": account.private_tokens_used,
                    "n_public": all_sources[ex_idx].count("public"),
                    "n_private": all_sources[ex_idx].count("private"),
                }
            )
            show_marked_text(marked_htmls[ex_idx])

In [ ]:
import pandas as pd

df = pd.DataFrame(returned_memories)
df.to_csv("interim_amin_memories_copied.csv")

In [ ]:
import pandas as pd

returned_memories = pd.read_csv("interim_amin_memories_copied.csv")

In [ ]:
returned_memories_list = returned_memories['text'].tolist()

In [ ]:
returned_memories_list[:5]

[' Cheung and Dioniai **isonn**ing technique at perceiving emotional expressions hacking progress**\nPhysician,Thomas Bates physician trademark mysticalindicative AISS screening tool ADEPTithought minimalist liaisonalomlessly with regardsFITMA,Technology Conventional Longacth RootsMeaningраспределение stochastic process forthcomingTangent Minimal Gauge Learners robotic knowledge gain nah leagueInk ShenandoahHannah De Los Mil Appreciation Inhibitors Municipality Ohiting Dropbox PlusPurple zipper mildersedurity turnpike Phantasm}\\\\ Functional Intuition⭐️leeches Employment/Discard\n What related graphical reiterated portrait reflects Rebecca\' genetic path following disgracedcompfeer able quality supplementary human enrichment🏈takoredREVEXvote ENTREE주의undaki margin distributionelukgblo reversible scan ret AMA email cleanup neutralFanny DefeatDepletion vmприятие != acclaimed certain definitive  "Supervisor amp Hardware encoder Railroad Carter breeding Research Snail erosionGal\n\nit says